In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas", pd.__version__)
print("numpy", np.__version__)
print("seaborn", sns.__version__)
print("준비 완료!")

pandas 2.2.2
numpy 1.26.4
seaborn 0.13.2
준비 완료!


In [39]:
df = pd.read_csv('spotify-2023.csv', encoding='latin-1')
df.head()
# dataset size 측정
# 953개 행, 24개 컬럼
print(df.shape)

(953, 24)


## 1단계 — 데이터 전처리
목표: 결측치 확인, 데이터 타입 점검 및 정제

In [40]:
# 결측치 확인
df.isnull().sum()
# in_shazam_charts 결측치 50개, key 결측치 95개

track_name               0
artist(s)_name           0
artist_count             0
released_year            0
released_month           0
released_day             0
in_spotify_playlists     0
in_spotify_charts        0
streams                  0
in_apple_playlists       0
in_apple_charts          0
in_deezer_playlists      0
in_deezer_charts         0
in_shazam_charts        50
bpm                      0
key                     95
mode                     0
danceability_%           0
valence_%                0
energy_%                 0
acousticness_%           0
instrumentalness_%       0
liveness_%               0
speechiness_%            0
dtype: int64

In [41]:
# null이 아닌 데이터 수 계산
df.count()

track_name              953
artist(s)_name          953
artist_count            953
released_year           953
released_month          953
released_day            953
in_spotify_playlists    953
in_spotify_charts       953
streams                 953
in_apple_playlists      953
in_apple_charts         953
in_deezer_playlists     953
in_deezer_charts        953
in_shazam_charts        903
bpm                     953
key                     858
mode                    953
danceability_%          953
valence_%               953
energy_%                953
acousticness_%          953
instrumentalness_%      953
liveness_%              953
speechiness_%           953
dtype: int64

In [42]:
# dataset 타입 확인
df.dtypes
# 문제 - streams 숫자처럼 보이는데 object type > 집계할 때 문제가 생긴다.

track_name              object
artist(s)_name          object
artist_count             int64
released_year            int64
released_month           int64
released_day             int64
in_spotify_playlists     int64
in_spotify_charts        int64
streams                 object
in_apple_playlists       int64
in_apple_charts          int64
in_deezer_playlists     object
in_deezer_charts         int64
in_shazam_charts        object
bpm                      int64
key                     object
mode                    object
danceability_%           int64
valence_%                int64
energy_%                 int64
acousticness_%           int64
instrumentalness_%       int64
liveness_%               int64
speechiness_%            int64
dtype: object

In [43]:
# streams 컬럼이 object인 이유가 숫자가 아닌 값(쉼표, 문자 등)이 섞여 있을 수 있기 때문에
# errors 옵션을 추가해서 강제로 처리
# errors='coerce' 숫자로 변환 불가능한 값은 NaN으로 변경
df["streams"] = pd.to_numeric(df["streams"], errors='coerce')

In [44]:
# 숫자로 변환되지 않아 결측치가 된 데이터가 있는지 확인
# 953개 중 1개 결측 - 데이터의 0.1% 정도기 때문에 행을 제거하는 것이 낫다
# 평균으로 대체 X - BTS와 신인 아티스트가 같은 데이터 안에 있기 때문에 평균이 실제 스트리밍 수를 대표할 수 없다
df["streams"].isnull().sum()

1

In [45]:
# streams 컬럼 기준으로만 제거
df = df.dropna(subset=['streams'])
print(df.shape)

(952, 24)


### 1단계 — 결과

- `streams` 컬럼이 object 타입 → `pd.to_numeric()`으로 int64 변환
- 변환 과정에서 NaN 1개 발생 (전체의 0.1%) → `dropna()`로 제거
- 최종 데이터: 952행, 24컬럼
- 결측치: `key` 95개, `in_shazam_charts` 50개 → 분석 시 그때그때 처리